In [2]:
import xarray as xr
import numpy as np
import glob
import os


In [3]:
input_folder  = "/home/lacrio/DATA/term_cal/data_700/"
input_folder1 = "/home/lacrio/DATA/term_cal/data_750_650/"
output_folder = "/home/lacrio/DATA/term_cal/res_new_700_sergi_case/"

os.makedirs(output_folder, exist_ok=True)

# Read files from both folders
files  = sorted(glob.glob(os.path.join(input_folder, "*.nc")))
files1 = sorted(glob.glob(os.path.join(input_folder1, "*.nc")))


In [16]:
# ============================================
# LOAD
# ============================================
ds = xr.open_dataset(files[92]).resample(valid_time="1D").mean()
ds = ds.rename({"valid_time": "time"})

ds1 = xr.open_dataset(files1[92]).resample(valid_time="1D").mean()
ds1 = ds1.rename({"valid_time": "time"})

# ============================================
# GET DIMENSIONS
# ============================================
lons = ds['longitude'].values
lats = ds['latitude'].values
times = ds['time'].values
times

array(['2021-02-04T00:00:00.000000000', '2021-02-05T00:00:00.000000000',
       '2021-02-06T00:00:00.000000000', '2021-02-07T00:00:00.000000000',
       '2021-02-08T00:00:00.000000000', '2021-02-09T00:00:00.000000000',
       '2021-02-10T00:00:00.000000000', '2021-02-11T00:00:00.000000000',
       '2021-02-12T00:00:00.000000000', '2021-02-13T00:00:00.000000000',
       '2021-02-14T00:00:00.000000000'], dtype='datetime64[ns]')

In [22]:
# ============================================
# VARIABLES (igual que tu código)
# ============================================
T900 = ds1['t'][:,0,:,:].values - 273.15
T950 = ds1['t'][:,1,:,:].values - 273.15
T925 = ds['t'][:,2,:,:].values - 273.15

w925 = ds['w'][:,2,:,:].values
U925 = ds['u'][:,2,:,:].values
V925 = ds['v'][:,2,:,:].values

# ============================================
# CALCULOS (SIN CAMBIOS)
# ============================================
lons_g, lats_g = np.meshgrid(lons, lats)

Xdist_arr = np.diff(lons_g * np.pi * 6371000 / 180) * np.cos(lats_g[:,:-1] * np.pi / 180)
Ydist_arr = np.diff(lats_g, axis=0) * np.pi * 6371000 / 180

dT_dx = (np.diff(T925, axis=2)[:,:,:-1] + np.diff(T925, axis=2)[:,:,1:]) / 2 / (2*Xdist_arr[:,:-1])
dT_dy = (np.diff(T925, axis=1)[:,:-1,:] + np.diff(T925, axis=1)[:,1:,:]) / 2 / (2*Ydist_arr[:-1,:])

Adv = - (
    U925[:,1:-1,1:-1] * dT_dx[:,1:-1,:] +
    V925[:,1:-1,1:-1] * dT_dy[:,:,1:-1]
) * 86400

Tpot950 = (T950+273.15)*(1000/750)**0.286
Tpot925 = (T925+273.15)*(1000/700)**0.286
Tpot900 = (T900+273.15)*(1000/850)**0.286

dTpot_dp = (Tpot950 - Tpot900) / (70000-85000)

VAdv = - (
    w925[:,1:-1,1:-1]
    * (T925[:,1:-1,1:-1]+273.15)
    / Tpot925[:,1:-1,1:-1]
    * dTpot_dp[:,1:-1,1:-1]
) * 86400

T925_sec = T925[:,1:-1,1:-1]

# ============================================
# LOOP TIEMPO (SIN CAMBIOS)
# ============================================
nt = len(times)

Adv_24H  = np.empty((nt, len(lats)-2, len(lons)-2))
VAdv_24H = np.empty((nt, len(lats)-2, len(lons)-2))
Tend_24H = np.empty((nt, len(lats)-2, len(lons)-2))
Res_24H  = np.empty((nt, len(lats)-2, len(lons)-2))
T925_24H = np.empty((nt, len(lats)-2, len(lons)-2))

for i in range(nt-1):
    Adv_24H[i]  = Adv[i]
    VAdv_24H[i] = VAdv[i]

    Tend_24H[i] = T925_sec[i+1] - T925_sec[i]
    Res_24H[i]  = Tend_24H[i] - Adv_24H[i] - VAdv_24H[i]

    T925_24H[i] = T925_sec[i]


# ============================================
# CREATE DATASET
# ============================================
lats_trim = lats[1:-1]
lons_trim = lons[1:-1]
times_trim = times[:-1]

ds_out = xr.Dataset(
    {
        "tendency": (("time","latitude","longitude"), Tend_24H[:-1]),
        "adv_h": (("time","latitude","longitude"), Adv_24H[:-1]),
        "adv_v": (("time","latitude","longitude"), VAdv_24H[:-1]),
        "residual": (("time","latitude","longitude"), Res_24H[:-1]),
        "t700": (("time","latitude","longitude"), T925_24H[:-1]),
    },
    coords={
        "time": times_trim,
        "latitude": lats_trim,
        "longitude": lons_trim,
    }
)

# metadata
ds_out["tendency"].attrs["units"] = "degC/day"
ds_out["adv_h"].attrs["units"]    = "degC/day"
ds_out["adv_v"].attrs["units"]    = "degC/day"
ds_out["residual"].attrs["units"] = "degC/day"
ds_out["t700"].attrs["units"]     = "degC"

ds_out.attrs["description"] = "Thermodynamic budget"
ds_out.attrs["method"] = "central differences + forward time (24h)"


In [23]:
ds_out

<xarray.Dataset> Size: 8MB
Dimensions:    (time: 10, latitude: 111, longitude: 179)
Coordinates:
  * time       (time) datetime64[ns] 80B 2021-02-04 2021-02-05 ... 2021-02-13
  * latitude   (latitude) float64 888B -42.25 -42.5 -42.75 ... -69.5 -69.75
  * longitude  (longitude) float64 1kB -84.75 -84.5 -84.25 ... -40.5 -40.25
Data variables:
    tendency   (time, latitude, longitude) float64 2MB -0.5801 -0.553 ... -2.801
    adv_h      (time, latitude, longitude) float64 2MB -0.07311 ... -0.7872
    adv_v      (time, latitude, longitude) float64 2MB 0.5722 0.7455 ... -0.9233
    residual   (time, latitude, longitude) float64 2MB -1.079 -1.293 ... -1.091
    t700       (time, latitude, longitude) float64 2MB 6.813 6.825 ... -12.91
Attributes:
    description:  Thermodynamic budget
    method:       central differences + forward time (24h)